# Simulasi Prediksi Penjualan Toko Ritel
Kelompok 1 - LM01
Louis Huang - Gilbert Tjandra Adanarianto - Dava Rabbani Adrian Widyatmoko

Tujuan: Membangun model Machine Learning berbasis regresi untuk memprediksi  
indeks penjualan toko ritel harian berdasarkan promosi, hari libur, ukuran toko, dan kondisi operasional.

Ciri-ciri model : 
- Model dilatih menggunakan dataset Rossman Store Sales (ritel Jerman).  
- Pola yang dipelajari (efek promosi, hari libur, persaingan, musim) bersifat universal dan digunakan sebagai simuilasi 
untuk toko ritel di Indonesia yang terlihat di angka/jumlah prediksi yang menggunakan Rupiah

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# Konversi EUR → IDR (perbarui sesuai kurs terkini)
EUR_TO_IDR = 20_759

print('Library berhasil dimuat!')

## 2. Load Dataset

In [ ]:
train_df = pd.read_csv('./train.csv', low_memory=False)
store_df  = pd.read_csv('./store.csv')

print('Train shape:', train_df.shape)
print('Store shape:', store_df.shape)
train_df.head()

In [ ]:
store_df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Info Dataset Train')
train_df.info()
print()
print('Missing Values - Train')
print(train_df.isnull().sum())

In [ ]:
print('=== Info Dataset Store ===')
store_df.info()
print()
print('=== Missing Values - Store ===')
print(store_df.isnull().sum())

In [ ]:
train_df.describe()

In [ ]:
# Distribusi Sales dan variabel numerik utama
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
open_df = train_df[train_df['Open'] == 1]

sns.histplot(open_df['Sales'], bins=50, kde=True, ax=axes[0][0], color='steelblue')
axes[0][0].set_title('Distribusi Penjualan Harian (Toko Buka)')
axes[0][0].set_xlabel('Penjualan (EUR — data latih)')

sns.histplot(open_df['Customers'], bins=50, kde=True, ax=axes[0][1], color='orange')
axes[0][1].set_title('Distribusi Jumlah Pelanggan Harian')
axes[0][1].set_xlabel('Jumlah Pelanggan')

sales_by_promo = open_df.groupby('Promo')['Sales'].mean().reset_index()
axes[1][0].bar(['Tanpa Promosi', 'Dengan Promosi'], sales_by_promo['Sales'], color=['#e74c3c', '#2ecc71'])
axes[1][0].set_title('Rata-rata Penjualan: Promosi vs Tidak')
axes[1][0].set_ylabel('Penjualan')

sales_by_dow = open_df.groupby('DayOfWeek')['Sales'].mean()
axes[1][1].bar(sales_by_dow.index, sales_by_dow.values, color='mediumpurple')
axes[1][1].set_title('Rata-rata Penjualan per Hari')
axes[1][1].set_xlabel('Hari (1=Senin, 7=Minggu)')
axes[1][1].set_ylabel('Penjualan')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: Pengaruh Hari Libur terhadap Penjualan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='StateHoliday', y='Sales', data=open_df, ax=axes[0], palette='Set2')
axes[0].set_title('Penjualan vs Hari Libur Nasional')
axes[0].set_xlabel('Hari Libur (0=Biasa, a=Publik, b=Easter, c=Natal)')

sns.boxplot(x='SchoolHoliday', y='Sales', data=open_df, ax=axes[1], palette='Set3')
axes[1].set_title('Penjualan vs Libur Sekolah')
axes[1].set_xlabel('Libur Sekolah (0=Tidak, 1=Ya)')

plt.tight_layout()
plt.show()

In [ ]:
# Distribusi Ukuran Toko dan Ragam Produk
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

store_labels = {0:'Kecil', 1:'Besar', 2:'Spesialis', 3:'Sangat Besar'}
store_df['StoreType'].map({'a':0,'b':1,'c':2,'d':3}).map(store_labels).value_counts().plot(
    kind='bar', ax=axes[0], color='coral', rot=0)
axes[0].set_title('Distribusi Ukuran Toko')
axes[0].set_xlabel('Ukuran Toko')

asst_labels = {0:'Basic', 1:'Standard', 2:'Lengkap'}
store_df['Assortment'].map({'a':0,'b':1,'c':2}).map(asst_labels).value_counts().plot(
    kind='bar', ax=axes[1], color='teal', rot=0)
axes[1].set_title('Distribusi Ragam Produk')
axes[1].set_xlabel('Ragam Produk')

plt.tight_layout()
plt.show()

## 4. Preprocessing Data

In [ ]:
# Gabungkan data train dan store
df = pd.merge(train_df, store_df, on='Store', how='left')
print('Shape setelah merge:', df.shape)

# Filter: hapus toko tutup (Sales=0 saat Open=0 adalah bias)
df = df[(df['Open'] == 1) & (df['Sales'] > 0)]
print('Setelah filter toko tutup:', df.shape)

In [ ]:
# Fitur temporal dari tanggal
df['Date'] = pd.to_datetime(df['Date'])
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Day']        = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)

print('Fitur tanggal ditambahkan.')
df[['Date','Year','Month','Day','WeekOfYear']].head()

In [ ]:
# Handle missing values
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())
df['CompetitionOpenSinceMonth'] = df['CompetitionOpenSinceMonth'].fillna(0)
df['CompetitionOpenSinceYear'] = df['CompetitionOpenSinceYear'].fillna(0)
df['Promo2SinceWeek'] = df['Promo2SinceWeek'].fillna(0)
df['Promo2SinceYear'] = df['Promo2SinceYear'].fillna(0)
df['PromoInterval'] = df['PromoInterval'].fillna('None')

print('Missing values setelah treatment:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Encoding kategorikal
holiday_map = {'0':0, 0:0, 'a':1, 'b':2, 'c':3}
df['StateHoliday'] = df['StateHoliday'].map(holiday_map).fillna(0).astype(int)

# StoreType: a=Kecil(0), b=Besar(1), c=Spesialis(2), d=Sangat Besar(3)
df['StoreType']   = df['StoreType'].map({'a':0,'b':1,'c':2,'d':3}).fillna(0).astype(int)

# Assortment: a=Basic(0), b=Standard(1), c=Lengkap(2)
df['Assortment']  = df['Assortment'].map({'a':0,'b':1,'c':2}).fillna(0).astype(int)

promo_map = {'None':0,'Jan,Apr,Jul,Oct':1,'Feb,May,Aug,Nov':2,'Mar,Jun,Sept,Dec':3}
df['PromoInterval'] = df['PromoInterval'].map(promo_map).fillna(0).astype(int)

print('Encoding selesai.')
print()
print('Mapping Ukuran Toko  : a→Kecil(0), b→Besar(1), c→Spesialis(2), d→Sangat Besar(3)')
print('Mapping Ragam Produk : a→Basic(0), b→Standard(1), c→Lengkap(2)')

In [ ]:
# Pilih fitur untuk model
FEATURES = [
    'Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday',
    'StoreType', 'Assortment', 'CompetitionDistance',
    'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
    'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval',
    'Year', 'Month', 'Day', 'WeekOfYear'
]
TARGET = 'Sales'

X = df[FEATURES]
y = np.log1p(df[TARGET])  # log transform untuk normalisasi distribusi

print('Shape X:', X.shape)
X.head()

In [ ]:
# Visualisasi distribusi Sales sebelum & sesudah log transform
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df[TARGET], bins=50, kde=True, ax=axes[0], color='salmon')
axes[0].set_title('Distribusi Penjualan (Original)')
axes[0].set_xlabel('Sales (data latih)')

sns.histplot(y, bins=50, kde=True, ax=axes[1], color='steelblue')
axes[1].set_title('Distribusi Penjualan (Log Transform)')
axes[1].set_xlabel('log1p(Sales)')

plt.tight_layout()
plt.show()

## 5. Pembagian Data (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training set : {X_train.shape[0]:,} baris')
print(f'Test set     : {X_test.shape[0]:,} baris')

## 6. Pelatihan Model

### 6a. Baseline: Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print('Linear Regression selesai dilatih.')

### 6b. Model Utama: Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=15,
    min_samples_split=5, min_samples_leaf=2,
    n_jobs=-1, random_state=42
)
rf_model.fit(X_train, y_train)
print('Random Forest selesai dilatih.')

### 6c. Model Pembanding: XGBoost Regressor

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, verbosity=0
)
xgb_model.fit(X_train, y_train)
print('XGBoost selesai dilatih.')

## 7. Evaluasi Model

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred_log  = model.predict(X_test)
    y_test_real = np.expm1(y_test)
    y_pred_real = np.expm1(y_pred_log)

    mae  = mean_absolute_error(y_test_real, y_pred_real)
    rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
    r2   = r2_score(y_test_real, y_pred_real)

    mean_sales = np.mean(y_test_real)
    mae_pct    = (mae  / mean_sales) * 100
    rmse_pct   = (rmse / mean_sales) * 100

    print(f'--- {name} ---')
    print(f'MAE  : {mae:,.2f}  (~{mae_pct:.2f}% dari rata-rata)')
    print(f'RMSE : {rmse:,.2f} (~{rmse_pct:.2f}% dari rata-rata)')
    print(f'R²   : {r2:.4f}')
    print(f'[IDR] MAE  ≈ Rp {mae*EUR_TO_IDR:,.0f}')
    print(f'[IDR] RMSE ≈ Rp {rmse*EUR_TO_IDR:,.0f}')
    print()

    return {
        'name': name, 'mae': mae, 'rmse': rmse, 'r2': r2,
        'mae_pct': mae_pct, 'rmse_pct': rmse_pct,
        'y_aktual':   y_test_real.values if hasattr(y_test_real,'values') else np.array(y_test_real),
        'y_prediksi': y_pred_real
    }

lr_eval  = evaluate_model('Linear Regression', lr_model,  X_test, y_test)
rf_eval  = evaluate_model('Random Forest',     rf_model,  X_test, y_test)
xgb_eval = evaluate_model('XGBoost',           xgb_model, X_test, y_test)

In [ ]:
# Perbandingan metrik
models_list = ['Linear Regression', 'Random Forest', 'XGBoost']
mae_vals    = [lr_eval['mae'],  rf_eval['mae'],  xgb_eval['mae']]
rmse_vals   = [lr_eval['rmse'], rf_eval['rmse'], xgb_eval['rmse']]
r2_vals     = [lr_eval['r2'],   rf_eval['r2'],   xgb_eval['r2']]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db']

axes[0].bar(models_list, mae_vals,  color=colors)
axes[0].set_title('MAE (lebih kecil = lebih baik)')
axes[0].tick_params(axis='x', rotation=10)

axes[1].bar(models_list, rmse_vals, color=colors)
axes[1].set_title('RMSE (lebih kecil = lebih baik)')
axes[1].tick_params(axis='x', rotation=10)

axes[2].bar(models_list, r2_vals,   color=colors)
axes[2].set_title('R² Score (lebih besar = lebih baik)')
axes[2].axhline(y=0.85, color='red', linestyle='--', label='Target R²=0.85')
axes[2].legend()
axes[2].tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Aktual vs Prediksi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, eval_data, title in zip(axes, [rf_eval, xgb_eval], ['Random Forest', 'XGBoost']):
    ax.scatter(eval_data['y_aktual'], eval_data['y_prediksi'], alpha=0.3, s=5, color='steelblue')
    mn = min(eval_data['y_aktual'].min(), eval_data['y_prediksi'].min())
    mx = max(eval_data['y_aktual'].max(), eval_data['y_prediksi'].max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Prediksi Sempurna')
    ax.set_xlabel('Penjualan Aktual (indeks)')
    ax.set_ylabel('Penjualan Prediksi (indeks)')
    ax.set_title(f'{title}: Aktual vs Prediksi (R²={eval_data["r2"]:.4f})')
    ax.legend()
plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
# Label ramah bisnis untuk fitur
FEAT_LABELS = {
    'Store':'Profil Toko', 'DayOfWeek':'Hari dalam Seminggu',
    'Promo':'Promosi Aktif', 'StateHoliday':'Hari Libur Nasional',
    'SchoolHoliday':'Libur Sekolah', 'StoreType':'Ukuran Toko',
    'Assortment':'Ragam Produk', 'CompetitionDistance':'Jarak ke Pesaing',
    'CompetitionOpenSinceMonth':'Bulan Buka Pesaing',
    'CompetitionOpenSinceYear':'Tahun Buka Pesaing',
    'Promo2':'Promo Berulang', 'Promo2SinceWeek':'Mulai Promo (Minggu)',
    'Promo2SinceYear':'Mulai Promo (Tahun)',
    'PromoInterval':'Musim Promo', 'Year':'Tahun',
    'Month':'Bulan', 'Day':'Tanggal', 'WeekOfYear':'Minggu ke-',
}

feat_imp = pd.Series(rf_model.feature_importances_, index=FEATURES)
feat_imp.index = [FEAT_LABELS.get(f, f) for f in feat_imp.index]
feat_imp = feat_imp.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='steelblue')
plt.title('Faktor yang Mempengaruhi Prediksi Penjualan')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 5 Faktor Paling Berpengaruh:')
print(feat_imp.head(5))

## 9. Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
best_r2   = max(rf_eval['r2'], xgb_eval['r2'])
best_name = 'Random Forest' if rf_eval['r2'] >= xgb_eval['r2'] else 'XGBoost'
print(f'Model terbaik untuk tuning: {best_name} (R²={best_r2:.4f})')

In [ ]:
sample_size = min(50000, len(X_train))
idx = np.random.choice(len(X_train), sample_size, replace=False)
X_sample = X_train.iloc[idx]
y_sample = y_train.iloc[idx]

if best_name == 'Random Forest':
    param_grid = {
        'n_estimators':[100,200,300], 'max_depth':[10,15,20,None],
        'min_samples_split':[2,5,10], 'min_samples_leaf':[1,2,4]
    }
    base_model = RandomForestRegressor(n_jobs=-1, random_state=42)
else:
    param_grid = {
        'n_estimators':[200,300,500], 'learning_rate':[0.01,0.05,0.1],
        'max_depth':[5,7,9], 'subsample':[0.7,0.8,0.9]
    }
    base_model = XGBRegressor(n_jobs=-1, random_state=42, verbosity=0)

tuner = RandomizedSearchCV(
    base_model, param_distributions=param_grid,
    n_iter=10, cv=3, scoring='r2', n_jobs=-1, random_state=42, verbose=1
)
tuner.fit(X_sample, y_sample)
print('Best params:', tuner.best_params_)
print(f'Best CV R²: {tuner.best_score_:.4f}')

In [ ]:
best_model = tuner.best_estimator_.__class__(**tuner.best_params_, n_jobs=-1, random_state=42)
best_model.fit(X_train, y_train)
tuned_eval = evaluate_model(f'{best_name} (Tuned)', best_model, X_test, y_test)

base_eval = rf_eval if best_name == 'Random Forest' else xgb_eval
print('Perbandingan sebelum dan sesudah tuning:')
print(f'Sebelum — R²: {base_eval["r2"]:.4f}, MAE: {base_eval["mae"]:,.0f}')
print(f'Sesudah — R²: {tuned_eval["r2"]:.4f}, MAE: {tuned_eval["mae"]:,.0f}')

## 10. Simpan Model & Artefak

In [ ]:
final_model = best_model
final_eval  = tuned_eval

joblib.dump(final_model, 'model_rossman.pkl')
joblib.dump(FEATURES,    'model_features.pkl')

# Feature importance dengan label internal (app.py yang menerjemahkan ke bahasa bisnis)
if hasattr(final_model, 'feature_importances_'):
    feat_imp_dict   = dict(zip(FEATURES, final_model.feature_importances_))
    feat_imp_sorted = dict(sorted(feat_imp_dict.items(), key=lambda x: x[1], reverse=True))
else:
    feat_imp_sorted = {}
joblib.dump(feat_imp_sorted, 'feature_importance.pkl')

eval_package = {
    'y_aktual':   final_eval['y_aktual'],
    'y_prediksi': final_eval['y_prediksi'],
    'mae':        final_eval['mae'],
    'rmse':       final_eval['rmse'],
    'r2':         final_eval['r2'],
    'mae_pct':    final_eval['mae_pct'],
    'rmse_pct':   final_eval['rmse_pct'],
    'model_name': f"{best_name} (Tuned)"
}
joblib.dump(eval_package, 'eval_data.pkl')

print('✅ Semua artefak berhasil disimpan:')
print('   - model_rossman.pkl')
print('   - model_features.pkl')
print('   - feature_importance.pkl')
print('   - eval_data.pkl')
print()
print(f'Kurs yang digunakan: 1 EUR = Rp {EUR_TO_IDR:,}')
print(f'MAE dalam Rupiah  ≈ Rp {final_eval["mae"] * EUR_TO_IDR:,.0f}')
print(f'RMSE dalam Rupiah ≈ Rp {final_eval["rmse"] * EUR_TO_IDR:,.0f}')